# Demo 08: Explanation Drift Under Shift

## Learning goals
- show that a prediction can stay stable while the explanation moves;
- compare a shortcut-sensitive classifier with a more stable intervention;
- visualise explanation drift directly inside the notebook.

## Why this demo matters
Headline accuracy or a single predicted label can hide explanation drift. The model may keep saying the same thing while relying on a different, less trustworthy cue.
## Data source
This notebook is self-contained. The base panel and all perturbations are generated inside the notebook; no external dataset files are read.



### Notebook display helpers

In [ ]:
import math

import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont

try:
    from IPython.display import display
except Exception:
    def display(obj):
        return None

FONT = ImageFont.load_default()


def np_to_pil(array):
    if isinstance(array, Image.Image):
        return array
    if array.dtype != np.uint8:
        array = (np.clip(array, 0.0, 1.0) * 255).astype(np.uint8)
    return Image.fromarray(array)


def titled(image, title, *, width=None):
    base = np_to_pil(image).convert('RGB')
    if width is not None and base.width != width:
        base = base.resize((width, round(base.height * width / base.width)))
    title_height = 22
    panel = Image.new('RGB', (base.width, base.height + title_height), (255, 255, 255))
    panel.paste(base, (0, title_height))
    draw = ImageDraw.Draw(panel)
    draw.rectangle((0, 0, base.width, title_height), fill=(245, 245, 245))
    draw.text((6, 5), title, fill=(20, 20, 20), font=FONT)
    return panel


def montage(panels, *, cols=3, padding=8, bg=(255, 255, 255)):
    prepared = [np_to_pil(panel).convert('RGB') for panel in panels]
    if not prepared:
        return Image.new('RGB', (320, 120), bg)
    cell_w = max(panel.width for panel in prepared)
    cell_h = max(panel.height for panel in prepared)
    rows = math.ceil(len(prepared) / cols)
    canvas = Image.new(
        'RGB',
        (cols * cell_w + (cols + 1) * padding, rows * cell_h + (rows + 1) * padding),
        bg,
    )
    for idx, panel in enumerate(prepared):
        row, col = divmod(idx, cols)
        x = padding + col * (cell_w + padding)
        y = padding + row * (cell_h + padding)
        canvas.paste(panel, (x, y))
    return canvas


def maybe_display(image):
    display(image)
    return image


def draw_box(image_array, box, *, colour=(220, 48, 48), width=3):
    image = np_to_pil(image_array).convert('RGB')
    draw = ImageDraw.Draw(image)
    x, y, w, h = box
    for offset in range(width):
        draw.rectangle((x - offset, y - offset, x + w + offset, y + h + offset), outline=colour)
    return image


def text_panel(lines, title, *, size=(560, 220)):
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    y = 38
    for line in lines:
        draw.text((12, y), line, fill=(20, 20, 20), font=FONT)
        y += 16
    return image


def bar_chart(values, labels, title, *, colours=None, size=(560, 280), value_fmt='{:.2f}'):
    if colours is None:
        colours = [(42, 157, 143)] * len(values)
    width, height = size
    left, right, top, bottom = 60, 20, 40, 50
    chart_w = width - left - right
    chart_h = height - top - bottom
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    max_value = max(max(values), 1e-6)
    bar_w = max(12, chart_w // max(len(values) * 2, 2))
    gap = bar_w
    x = left
    baseline = top + chart_h
    draw.line((left, top, left, baseline), fill=(60, 60, 60), width=1)
    draw.line((left, baseline, left + chart_w, baseline), fill=(60, 60, 60), width=1)
    for value, label, colour in zip(values, labels, colours, strict=True):
        bar_h = int((value / max_value) * (chart_h - 10)) if max_value > 0 else 0
        draw.rectangle((x, baseline - bar_h, x + bar_w, baseline), fill=colour)
        draw.text((x, baseline + 6), label[:12], fill=(20, 20, 20), font=FONT)
        draw.text((x, baseline - bar_h - 14), value_fmt.format(value), fill=(20, 20, 20), font=FONT)
        x += bar_w + gap
    return image


def grouped_bar_chart(groups, series_names, matrix, title, *, colours=None, size=(680, 300), value_fmt='{:.2f}'):
    if colours is None:
        colours = [(164, 68, 68), (43, 122, 120), (69, 123, 157)]
    width, height = size
    left, right, top, bottom = 60, 20, 40, 55
    chart_w = width - left - right
    chart_h = height - top - bottom
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    max_value = max(max(series) for series in matrix)
    baseline = top + chart_h
    draw.line((left, top, left, baseline), fill=(60, 60, 60), width=1)
    draw.line((left, baseline, left + chart_w, baseline), fill=(60, 60, 60), width=1)
    group_w = chart_w // max(len(groups), 1)
    series_count = len(series_names)
    bar_w = max(10, group_w // (series_count + 1))
    for group_idx, group in enumerate(groups):
        base_x = left + group_idx * group_w + 10
        for series_idx, _ in enumerate(series_names):
            value = matrix[series_idx][group_idx]
            bar_h = int((value / max(max_value, 1e-6)) * (chart_h - 12)) if max_value > 0 else 0
            x0 = base_x + series_idx * bar_w
            draw.rectangle((x0, baseline - bar_h, x0 + bar_w - 2, baseline), fill=colours[series_idx % len(colours)])
            draw.text((x0, baseline - bar_h - 12), value_fmt.format(value), fill=(20, 20, 20), font=FONT)
        draw.text((base_x, baseline + 6), group[:12], fill=(20, 20, 20), font=FONT)
    legend_x = width - 210
    for idx, name in enumerate(series_names):
        y = 24 + idx * 16
        draw.rectangle((legend_x, y, legend_x + 10, y + 10), fill=colours[idx % len(colours)])
        draw.text((legend_x + 16, y - 1), name, fill=(20, 20, 20), font=FONT)
    return image

### Synthetic image and perturbation code

In [ ]:
def render_defect_panel(*, stamp='red'):
    image = Image.new('RGB', (128, 128), (44, 52, 58))
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((20, 24, 108, 112), radius=6, fill=(92, 102, 104))
    draw.rounded_rectangle((44, 42, 86, 84), radius=2, fill=(188, 204, 196), outline=(32, 38, 40), width=3)
    colours = {'red': (216, 70, 64), 'blue': (56, 116, 214), 'none': (44, 52, 58)}
    draw.rectangle((6, 6, 26, 26), fill=colours[stamp])
    return image


def perturb(image, mode):
    out = image.copy()
    if mode == 'clean':
        pass
    elif mode == 'low_contrast':
        out = ImageEnhance.Contrast(out).enhance(0.65)
    elif mode == 'stamp_fade':
        draw = ImageDraw.Draw(out)
        draw.rectangle((6, 6, 26, 26), fill=(120, 96, 94))
    elif mode == 'blur':
        out = out.filter(ImageFilter.GaussianBlur(radius=1.2))
    else:
        raise ValueError(mode)
    return np.asarray(out, dtype=np.float32) / 255.0

### Two simple models and their contribution accounting

In [ ]:
def stamp_feature(image):
    region = image[6:26, 6:26]
    return float(region[:, :, 0].mean() - region[:, :, 2].mean())


def shape_feature(image):
    region = image[42:86, 42:86]
    occupied = np.linalg.norm(region - np.array([0.74, 0.80, 0.77]), axis=2) < 0.22
    row_counts = occupied.sum(axis=1)
    full_rows = np.count_nonzero(row_counts > 26)
    centre_column = np.count_nonzero(occupied[:, occupied.shape[1] // 2])
    return float(full_rows - 0.15 * centre_column - 18)


def model_contributions(image, *, stamp_weight, shape_weight):
    stamp = stamp_weight * stamp_feature(image)
    shape = shape_weight * shape_feature(image)
    total = stamp + shape
    denom = abs(stamp) + abs(shape) + 1e-6
    return {
        'stamp': float(stamp),
        'shape': float(shape),
        'prediction': 'defect' if total > 0 else 'normal',
        'total': float(total),
        'stamp_share': float(abs(stamp) / denom),
        'shape_share': float(abs(shape) / denom),
    }


def evidence_panel(image, mode, contribs):
    if contribs['stamp_share'] >= contribs['shape_share']:
        return titled(draw_box(image, (6, 6, 20, 20)), f'{mode} | stamp-led')
    return titled(draw_box(image, (42, 42, 44, 44)), f'{mode} | shape-led')

### Build the perturbation set and score both models

In [ ]:
base_image = render_defect_panel(stamp='red')
perturbations = {mode: perturb(base_image, mode) for mode in ['clean', 'low_contrast', 'stamp_fade', 'blur']}
shortcut_model = {mode: model_contributions(image, stamp_weight=1.15, shape_weight=0.25) for mode, image in perturbations.items()}
stable_model = {mode: model_contributions(image, stamp_weight=0.12, shape_weight=1.05) for mode, image in perturbations.items()}

## Dataset and task definition
We use a single synthetic defect panel and apply benign perturbations such as lower contrast, blur, and a faded corner stamp. The label remains `defect` throughout.

In [ ]:
maybe_display(montage([titled(image, mode) for mode, image in perturbations.items()], cols=4))

## Model and explanation methods
Two explicit models are compared.

- The shortcut-sensitive model leans heavily on the corner stamp.
- The intervention model leans mainly on the central part geometry.

Because the models are additive rules over two regions, we can measure explanation drift as a shift in contribution share between the stamp region and the shape region.

In [ ]:
maybe_display(montage([
    titled(draw_box(perturbations['clean'], (6, 6, 20, 20)), 'Shortcut-sensitive evidence region'),
    titled(draw_box(perturbations['clean'], (42, 42, 44, 44)), 'Intervention evidence region'),
], cols=2))

## Baseline result
The shortcut-sensitive model keeps predicting `defect` under all perturbations, so the prediction looks stable.

In [ ]:
maybe_display(montage([
    titled(image, f'{mode} -> {shortcut_model[mode]["prediction"]}')
    for mode, image in perturbations.items()
], cols=4))

## Failure or pitfall
The prediction is stable, but the explanation is not. Under perturbation, the shortcut-sensitive model shifts how much it relies on the stamp versus the central shape.

In [ ]:
maybe_display(grouped_bar_chart(
    list(perturbations),
    ['Stamp share', 'Shape share'],
    [[shortcut_model[mode]['stamp_share'] for mode in perturbations], [shortcut_model[mode]['shape_share'] for mode in perturbations]],
    'Shortcut-sensitive explanation drift',
))
maybe_display(montage([evidence_panel(image, mode, shortcut_model[mode]) for mode, image in perturbations.items()], cols=4))

## Intervention
The intervention is to use a model whose score is dominated by the object region rather than the nuisance region.

In [ ]:
maybe_display(grouped_bar_chart(
    list(perturbations),
    ['Stamp share', 'Shape share'],
    [[stable_model[mode]['stamp_share'] for mode in perturbations], [stable_model[mode]['shape_share'] for mode in perturbations]],
    'Intervention explanation stability',
))

## Re-test
After the intervention, the prediction is still stable, but now the explanation is also much more stable because the object region remains dominant under shift.

In [ ]:
maybe_display(montage([evidence_panel(image, mode, stable_model[mode]) for mode, image in perturbations.items()], cols=4))

## What we learned
- Stable predictions do not guarantee stable explanations.
- Explanation drift can reveal a nuisance-sensitive model even when the label does not flip.
- A more object-focused model can improve both robustness and interpretability.

## Residual risks and next questions
- This is a hand-crafted demonstration rather than a learned saliency model.
- Real explanation drift can appear in more subtle forms than a two-region contribution shift.
- A fuller version would track drift quantitatively across many images and perturbations.